# 2. Bring your own predictor as ONNX and explain it with SCE

This is the README section *"How to use a custom image dataset and a custom predictor"* as a
notebook.

Here PEAL does not train your classifier. You hand it an ONNX file, it is called forward only, and
SCE explains it. You still need a generator for the image space; this notebook uses the shipped
ImageNet DDPM config, which can be bootstrapped from 256x256 ImageNet weights.

**The one thing that goes wrong most often:** PEAL feeds the ONNX graph exactly the tensor your data
config produces, with no normalization of its own. Whatever preprocessing your model expects has to
be expressed in the data config's `input_size` and `normalization`.

In [ ]:
import os, subprocess, sys, textwrap
from pathlib import Path

# PEAL resolves <PEAL_BASE> itself; $PEAL_DATA and $PEAL_RUNS default to ./datasets and ./peal_runs.
PEAL_BASE = Path(os.environ.get("PEAL_BASE", Path.cwd().parent)).resolve()
PEAL_DATA = Path(os.environ.get("PEAL_DATA", PEAL_BASE / "datasets"))
PEAL_RUNS = Path(os.environ.get("PEAL_RUNS", PEAL_BASE / "peal_runs"))
os.chdir(PEAL_BASE)          # every entry point expects the repo root as the working directory
os.environ["PEAL_DATA"] = str(PEAL_DATA)
os.environ["PEAL_RUNS"] = str(PEAL_RUNS)
print("PEAL_BASE", PEAL_BASE, "\nPEAL_DATA", PEAL_DATA, "\nPEAL_RUNS", PEAL_RUNS)

def run(cmd):
    """Stream a PEAL command. These are long jobs; the output is the run log."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    if code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {cmd}")
    return code


In [ ]:
import yaml

def patch_yaml(template, target, **changes):
    """Copy a shipped config and overwrite single top-level keys.

    Comments are lost (yaml round-trips values only), which is why the README tells you to copy the
    files by hand. Doing it programmatically keeps a notebook reproducible; open the template if you
    want to read what each field means.
    """
    src = PEAL_BASE / template
    dst = PEAL_BASE / target
    dst.parent.mkdir(parents=True, exist_ok=True)
    cfg = yaml.safe_load(src.read_text())
    for k, v in changes.items():
        cfg[k] = v
    dst.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(f"wrote {target}")
    return dst


## Step 1 — export your model

A binary head is the simplest case. This is the pattern of
`tools/convert_imagenet_resnet18_to_binary_onnx.py`, which slices two ImageNet logits out of a torchvision
ResNet-18. Keep the dynamic batch axis: PEAL calls the graph in batches.

In [ ]:
import torch, torchvision

class BinaryImageNetModel(torch.nn.Module):
    """Example: turn a 1000-way ImageNet model into a two-class one."""
    def __init__(self, class1, class2):
        super().__init__()
        self.model = torchvision.models.resnet18(pretrained=True)
        self.class1, self.class2 = class1, class2
    def forward(self, x):
        return self.model(x)[:, [self.class1, self.class2]]

DATASET = "my_data"
model = BinaryImageNetModel(248, 269).eval()     # 248 eskimo dog, 269 wolf
onnx_path = PEAL_RUNS / DATASET / "classifier" / "model.onnx"
onnx_path.parent.mkdir(parents=True, exist_ok=True)
torch.onnx.export(
    model, torch.randn(1, 3, 224, 224), str(onnx_path),
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=11,
)
print("wrote", onnx_path)


## Step 2 — check the graph actually runs the way PEAL will call it

PEAL loads `.onnx` students through `load_onnx_predictor` (see
`peal/architectures/predictors.py` and `peal/architectures/onnx_predictor.py`): it converts the graph
to a trainable torch module with `onnx2torch` and falls back to an `onnxruntime` closure when an
operator is unsupported. Either way the dataset tensor is passed straight through. Reproduce that here before spending GPU hours on a run that
would fail at the first batch.

In [ ]:
# onnx / onnxruntime are NOT in the base `peal` environment: install the web extras
# (pip install -r requirements-web.txt) or use the .venv-web interpreter.
import numpy as np, onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
name = sess.get_inputs()[0].name
probe = np.random.rand(2, 3, 224, 224).astype(np.float32)
out = sess.run(None, {name: probe})[0]
print("input", sess.get_inputs()[0].shape, "-> output", out.shape)
assert out.shape[0] == 2, "the batch axis is not dynamic; re-export with dynamic_axes"


## Step 3 — dataset and configs

The csv header for this path is `ImagePath,<label>` (the SCE ImageNet configs use it, whereas the
notebook-1 path uses `imgs`). Set `normalization` to what your ONNX graph expects; `null` hands it
raw `[0, 1]` images.

### First, write the dataset PEAL will read

`Image2MixedDataset` wants `$PEAL_DATA/<name>/imgs/...` plus a `data.csv` whose first column is each
image's path relative to `imgs`, followed by one column per label. Notebook 1 has the same cell;
without it the config below points at a dataset that does not exist.


In [ ]:
import csv

SOURCE_DIR = PEAL_DATA / DATASET / "imgs"

def label_of(path):
    """Return 0 or 1 for an image path. Replace with your own rule."""
    return 1 if path.parent.name.lower().startswith("positive") else 0

rows = [(str(pp.relative_to(SOURCE_DIR)), label_of(pp))
        for pp in sorted(SOURCE_DIR.rglob("*"))
        if pp.suffix.lower() in {".png", ".jpg", ".jpeg"}]
if not rows:
    raise FileNotFoundError(f"no images under {SOURCE_DIR}")
csv_path = PEAL_DATA / DATASET / "data.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["imgs", "Label1"]); w.writerows(rows)
print(f"{len(rows)} images -> {csv_path}")


In [ ]:
# The Image2MixedDataset template that actually reads a data.csv. NOT
# configs/sce_experiments/data/imagenet.yaml, which is an ImageFolder loader with output_size [1000]
# and would ignore the csv written above.
data_cfg = patch_yaml(
    "configs/sce_experiments/data/imagenet_husky_vs_wulf_resnet18.yaml",
    f"configs/my_experiments/data/{DATASET}.yaml",
    dataset_path=f"$PEAL_DATA/{DATASET}",
    input_size=[3, 224, 224],
    normalization=[[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]],   # match YOUR model
)
cfg = yaml.safe_load(data_cfg.read_text())
cfg.pop("dataset_class", None); cfg.pop("confounding_factors", None)
data_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))

# task.y_selection names the label column. It is nested under `task:`; a top-level y_selection is
# silently ignored, and the template would keep looking for the husky/wulf label "IsWulf".
patch_yaml("configs/sce_experiments/adaptors/imagenet_husky_vs_wulf_sce_cfkd.yaml",
           f"configs/my_experiments/adaptors/{DATASET}_sce_cfkd.yaml",
           data=f"<PEAL_BASE>/configs/my_experiments/data/{DATASET}.yaml",
           student=f"$PEAL_RUNS/{DATASET}/classifier/model.onnx",
           base_dir=f"$PEAL_RUNS/{DATASET}/classifier/sce_cfkd",
           task={"criterions": {"ce": 1.0}, "output_type": "singleclass",
                 "output_channels": 2, "y_selection": ["Label1"]})


### Optional: your own generator instead of the ImageNet DDPM

Skip this if the shipped ImageNet DDPM suits your images. Otherwise derive a generator config and
train it — the expensive step. Drop `download_weights` if you do not want to bootstrap from the
ImageNet 256x256 weights.

In [ ]:
USE_OWN_GENERATOR = False
if USE_OWN_GENERATOR:
    p = patch_yaml("configs/sce_experiments/data/imagenet_generator.yaml",
                   f"configs/my_experiments/data/{DATASET}_generator.yaml",
                   dataset_path=f"$PEAL_DATA/{DATASET}")
    cfg = yaml.safe_load(p.read_text()); cfg.pop("dataset_class", None)
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))
    patch_yaml("configs/sce_experiments/generators/imagenet_ddpm.yaml",
               f"configs/my_experiments/generators/{DATASET}_ddpm.yaml",
               base_path=f"$PEAL_RUNS/{DATASET}/ddpm",
               data=f"<PEAL_BASE>/configs/my_experiments/data/{DATASET}_generator.yaml")
    run(f'python train_generator.py --config "<PEAL_BASE>/configs/my_experiments/generators/{DATASET}_ddpm.yaml"')
    patch_yaml(f"configs/my_experiments/adaptors/{DATASET}_sce_cfkd.yaml",
               f"configs/my_experiments/adaptors/{DATASET}_sce_cfkd.yaml",
               generator=f"$PEAL_RUNS/{DATASET}/ddpm/config.yaml")


## Step 4 — explain

In [ ]:
run(f'python run_cfkd.py --config "<PEAL_BASE>/configs/my_experiments/adaptors/{DATASET}_sce_cfkd.yaml"')

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def show_counterfactuals(npz_path, n=4, title=""):
    """Plot factual/counterfactual pairs out of a run's tracked values."""
    z = np.load(npz_path, allow_pickle=True)
    x  = np.asarray(z["x_list"], dtype=np.float32)
    cf = np.asarray(z["x_counterfactual_list"], dtype=np.float32)
    n = min(n, len(cf))
    fig, ax = plt.subplots(n, 2, figsize=(4, 2 * n))
    ax = np.atleast_2d(ax)
    img = lambda a: np.clip(a.transpose(1, 2, 0), 0, 1)
    for i in range(n):
        ax[i, 0].imshow(img(x[i])); ax[i, 0].set_ylabel(f"#{i}", fontsize=8)
        ax[i, 1].imshow(img(cf[i]))
        for j in (0, 1):
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        if i == 0:
            ax[i, 0].set_title("factual", fontsize=9)
            ax[i, 1].set_title("counterfactual", fontsize=9)
    fig.suptitle(title); fig.tight_layout(); plt.show()


In [ ]:
npz = PEAL_RUNS / DATASET / "classifier" / "sce_cfkd" / "0" / "validation_tracked_values.npz"
print("exists:", npz.exists())
if npz.exists():
    show_counterfactuals(npz, n=4, title="SCE counterfactuals for the ONNX model")
